# Q5 — Derive a Convolutional Network from a Design Principle

Implements the assignment's required fully connected versus CNN comparison, mathematical parameter derivations, translation test, learned filter visualisation and feature-map visualisation. Dataset: Fashion-MNIST.

## 1. Imports, reproducibility and runtime

In [ ]:
import os, time, random, json, zipfile, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED=42
EPOCHS=3
BATCH_SIZE=128
LR=0.001
TRAIN_N, VAL_N, TEST_N = 5000, 1000, 2000

def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

if not torch.cuda.is_available():
    torch.set_num_threads(min(4, os.cpu_count() or 1))
seed_all()
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT=Path("Q5_outputs"); OUT.mkdir(exist_ok=True)
print("Device:",device)
if device.type=="cuda": print(torch.cuda.get_device_name(0))


## 2. Dataset and fixed splits

Fashion-MNIST is used as the common image dataset. The same training, validation and test examples are used for both models. The reduced subset keeps execution manageable; describe this limitation in the report.

In [ ]:
tf=transforms.ToTensor()
train_full=datasets.FashionMNIST("./data",train=True,download=True,transform=tf)
test_full=datasets.FashionMNIST("./data",train=False,download=True,transform=tf)
g=torch.Generator().manual_seed(SEED)
perm=torch.randperm(len(train_full),generator=g)
train_ds=Subset(train_full,perm[:TRAIN_N].tolist())
val_ds=Subset(train_full,perm[TRAIN_N:TRAIN_N+VAL_N].tolist())
test_ds=Subset(test_full,list(range(TEST_N)))

def loaders(seed=SEED):
    gen=torch.Generator().manual_seed(seed)
    return (
      DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,generator=gen),
      DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0),
      DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    )
train_loader,val_loader,test_loader=loaders()
print("Train/validation/test:",len(train_ds),len(val_ds),len(test_ds))
print("Image shape:",train_full[0][0].shape,"Classes:",train_full.classes)


## 3. Design principle and mathematical derivations

A fully connected first layer maps all \(28\times28=784\) input values to 128 hidden units. Its parameter count is \(784(128)+128=100,480\).

A convolutional first layer with one input channel, 16 filters of size \(3\times3\), and one bias per filter has \(16(1\cdot3\cdot3)+16=160\) parameters.

For a convolution, the spatial output size is:

\[
O=\left\lfloor\frac{I+2P-K}{S}\right\rfloor+1
\]

For a 28×28 image, kernel 3, stride 1 and padding 1, the output remains 28×28. The same 3×3 detector is reused at every spatial position (weight sharing).

In [ ]:
# 3. Explicit parameter-count verification
IMAGE_PIXELS=28*28
FC_HIDDEN=128
FC_FIRST_PARAMS=IMAGE_PIXELS*FC_HIDDEN+FC_HIDDEN
CONV_OUT_CHANNELS=16
KERNEL=3
CONV_FIRST_PARAMS=CONV_OUT_CHANNELS*(1*KERNEL*KERNEL)+CONV_OUT_CHANNELS
print("FC first-layer parameters:",FC_FIRST_PARAMS)
print("CNN first-layer parameters:",CONV_FIRST_PARAMS)
print("Parameter reduction factor:",round(FC_FIRST_PARAMS/CONV_FIRST_PARAMS,2))

def conv_output_size(I,K=3,S=1,P=1):
    return (I+2*P-K)//S+1
print("Conv output size:",conv_output_size(28), "x", conv_output_size(28))


## 4. Define comparable fully connected and convolutional models

Both models receive the same 28×28 grayscale images and produce 10 class logits. The FC network uses flattened pixels; the CNN uses local filters and pooling.

In [ ]:
class FullyConnectedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Flatten(),nn.Linear(784,128),nn.ReLU(),nn.Linear(128,10))
    def forward(self,x): return self.net(x)

class DesignedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(1,16,3,padding=1)
        self.pool=nn.MaxPool2d(2)
        self.conv2=nn.Conv2d(16,32,3,padding=1)
        self.fc=nn.Linear(32*7*7,10)
    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))
        x=self.pool(F.relu(self.conv2(x)))
        return self.fc(torch.flatten(x,1))

factories={"FullyConnected":FullyConnectedNet,"CNN":DesignedCNN}
for name,factory in factories.items():
    m=factory()
    print(name,"total trainable parameters:",sum(p.numel() for p in m.parameters() if p.requires_grad))


## 5. Training and evaluation functions

The validation set monitors learning. The test set is evaluated once after each model has completed training. Training time includes the training epochs and validation passes.

In [ ]:
criterion=nn.CrossEntropyLoss()
def evaluate(model,loader):
    model.eval(); loss_sum=0; pred=[]; actual=[]
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(device),y.to(device)
            z=model(x); loss_sum+=criterion(z,y).item()*len(y)
            pred.extend(z.argmax(1).cpu().tolist()); actual.extend(y.cpu().tolist())
    return {"loss":loss_sum/len(loader.dataset),"accuracy":accuracy_score(actual,pred),
            "predictions":pred,"targets":actual}

def fit(model):
    tr,va,te=loaders()
    opt=optim.Adam(model.parameters(),lr=LR); hist=[]
    if device.type=="cuda": torch.cuda.synchronize()
    start=time.perf_counter()
    for ep in range(1,EPOCHS+1):
        model.train(); total=0
        for x,y in tr:
            x,y=x.to(device),y.to(device); opt.zero_grad(set_to_none=True)
            loss=criterion(model(x),y); loss.backward(); opt.step()
            total+=loss.item()*len(y)
        vm=evaluate(model,va)
        hist.append({"epoch":ep,"train_loss":total/len(tr.dataset),
                     "val_loss":vm["loss"],"val_accuracy":vm["accuracy"]})
        print(f"Epoch {ep}/{EPOCHS}: train loss={total/len(tr.dataset):.4f}, val loss={vm['loss']:.4f}, val acc={vm['accuracy']:.4f}")
    if device.type=="cuda": torch.cuda.synchronize()
    elapsed=time.perf_counter()-start
    test=evaluate(model,te)
    return hist,elapsed,test


## 6. Train and compare both models

In [ ]:
results=[]; histories={}; predictions={}
for name,factory in factories.items():
    print("\\nTraining",name); seed_all()
    model=factory().to(device)
    hist,elapsed,test=fit(model)
    params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    results.append({"Model":name,"Parameters":params,"Training_Time_s":elapsed,
                    "Test_Loss":test["loss"],"Test_Accuracy":test["accuracy"]})
    histories[name]=hist; predictions[name]=test
    torch.save(model.state_dict(),OUT/f"{name}_weights.pth")
    pd.DataFrame(results).to_csv(OUT/"Q5_results_checkpoint.csv",index=False)
    del model; gc.collect()
results_df=pd.DataFrame(results); display(results_df)
results_df.to_csv(OUT/"Q5_Final_Results.csv",index=False)


## 7. Translation test

A fixed translation is applied to each test image using zero-filled padding. Accuracy is measured on the original test images and on the translated images. This measures sensitivity to this particular shift; it does not establish invariance to every possible transformation.

In [ ]:
def translate_batch(x,dx=3,dy=2):
    # Positive dx shifts right; positive dy shifts down. Vacated pixels are zero-filled.
    theta=torch.tensor([[1,0,-2*dx/28],[0,1,-2*dy/28]],dtype=x.dtype,device=x.device)
    theta=theta.unsqueeze(0).repeat(x.size(0),1,1)
    grid=F.affine_grid(theta,x.size(),align_corners=False)
    return F.grid_sample(x,grid,mode="nearest",padding_mode="zeros",align_corners=False)

def evaluate_translation(model,loader):
    model.eval(); correct0=correct1=total=0
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(device),y.to(device)
            p0=model(x).argmax(1)
            shifted=translate_batch(x,dx=3,dy=2)
            p1=model(shifted).argmax(1)
            correct0+=(p0==y).sum().item(); correct1+=(p1==y).sum().item(); total+=len(y)
    return correct0/total,correct1/total

translation_rows={}
for name,factory in factories.items():
    model=factory().to(device)
    model.load_state_dict(torch.load(OUT/f"{name}_weights.pth",map_location=device))
    base,shift=evaluate_translation(model,test_loader)
    translation_rows[name]={"Original_Accuracy":base,"Shifted_Accuracy":shift,
                            "Accuracy_Drop":base-shift}
    print(name,translation_rows[name])
    del model
translation_df=pd.DataFrame(translation_rows).T
display(translation_df)
translation_df.to_csv(OUT/"Q5_translation_test.csv")


## 8. Visualise learned first-layer filters and feature maps

For the CNN, display its learned first-layer kernels and feature maps for one test image. A fully connected first layer does not have spatial convolutional filters, so filter visualisation applies to the CNN.

In [ ]:
cnn=DesignedCNN().to(device)
cnn.load_state_dict(torch.load(OUT/"CNN_weights.pth",map_location=device))
cnn.eval()
filters=cnn.conv1.weight.detach().cpu()
fig,axes=plt.subplots(2,8,figsize=(12,4))
for i,ax in enumerate(axes.flat):
    ax.imshow(filters[i,0],cmap="gray"); ax.set_title(f"Filter {i+1}"); ax.axis("off")
plt.suptitle("Learned first-layer CNN filters"); plt.tight_layout(); plt.show()

sample,label=test_ds[0]
x=sample.unsqueeze(0).to(device)
with torch.no_grad():
    fmap=F.relu(cnn.conv1(x)).cpu().squeeze(0)
fig,axes=plt.subplots(2,8,figsize=(12,4))
for i,ax in enumerate(axes.flat):
    ax.imshow(fmap[i],cmap="viridis"); ax.set_title(f"Map {i+1}"); ax.axis("off")
plt.suptitle(f"First-layer feature maps — true class: {train_full.classes[label]}")
plt.tight_layout(); plt.show()


## 9. Learning curves, reports and export

In [ ]:
plt.figure(figsize=(8,4))
for name,h in histories.items():
    plt.plot([r["epoch"] for r in h],[r["train_loss"] for r in h],marker="o",label=f"{name} train")
    plt.plot([r["epoch"] for r in h],[r["val_loss"] for r in h],marker="s",linestyle="--",label=f"{name} val")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss"); plt.title("Training and validation loss")
plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig(OUT/"Q5_loss_curves.png",dpi=150); plt.show()

for name,test in predictions.items():
    print("\\n",name)
    print(classification_report(test["targets"],test["predictions"],target_names=train_full.classes,zero_division=0))
    cm=confusion_matrix(test["targets"],test["predictions"])
    pd.DataFrame(cm,index=train_full.classes,columns=train_full.classes).to_csv(OUT/f"{name}_confusion_matrix.csv")

with open(OUT/"Q5_histories.json","w") as f: json.dump(histories,f,indent=2)
with open(OUT/"Q5_translation_results.json","w") as f: json.dump(translation_rows,f,indent=2)
with open(OUT/"Q5_settings.json","w") as f:
    json.dump({"seed":SEED,"epochs":EPOCHS,"batch_size":BATCH_SIZE,"learning_rate":LR,
               "train":TRAIN_N,"validation":VAL_N,"test":TEST_N,"device":str(device)},f,indent=2)

zipname="Q5_complete_evidence.zip"
with zipfile.ZipFile(zipname,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUT.parent))
print("Created",zipname)
try:
    from google.colab import files
    files.download(zipname)
except ImportError:
    print("ZIP saved in current working directory.")
